# iToF / Structured-Light Fusion Pipeline

Calibration and depth fusion for a combined indirect Time-of-Flight (iToF) and dot-projector sensor.  
Based on:
- **Microsoft-Paper** – Godbaz et al. 2025 (Microsoft): dot calibration, consistency error, active brightness trail
- **Agresti** – Agresti & Zanuttigh, ECCV 2018: maximum likelihood depth fusion

In [ ]:
from dot_calibration import DotCalibration

import os
import numpy as np
from pathlib import Path
import scipy.signal
from scipy.spatial import cKDTree
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

dotCal = DotCalibration()

# ── Data paths ──────────────────────────────────────────────────────────
base     = Path.cwd().parent / "Simulation_Pictures" / "PBRT" / "SL_ToF_1"
sl_cal   = base / "SL"           # calibration SL images (.exr)
tof_cal  = base / "ToF"          # calibration ToF point clouds (.pcd)
test_dir = base / "General_Test" # test scene

# ── Camera intrinsics ───────────────────────────────────────────────────
K     = np.array([[483.0,   0.0, 320.0],
                  [  0.0, 483.0, 240.0],
                  [  0.0,   0.0,   1.0]])
K_inv = np.linalg.inv(K)
fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]

print("Setup complete.")

## Calibration Steps 1–2: Dot Detection & Subpixel Localisation

LoG blob detection on each calibration SL image, followed by subpixel refinement.
Dots are assigned stable IDs 0–99 (top-left → bottom-right).

In [ ]:
# Collect calibration PCD paths sorted by distance
pcd_files = sorted(
    tof_cal.glob("SL_ToF_*m.pcd"),
    key=lambda p: dotCal.parse_distance_from_name(p.name) or 999.0
)
tof_paths = [str(p) for p in pcd_files]
cal_dists = [dotCal.parse_distance_from_name(p) for p in tof_paths]
print(f"Calibration distances: {cal_dists}")

subpixel_list = []

for dist, exr_path in zip(
        cal_dists,
        [str(sl_cal / f"SL_{d:.1f}m.exr") for d in cal_dists]
):
    print(f"  Processing {dist:.1f} m …")
    blobs, image = dotCal.detect_blobs(
        exr_path, max_sigma=25, num_sigma=10, threshold=0.01
    )
    _, subpixels = dotCal.detect_subpixel_locations(blobs, image, mode="center")
    subpixel_list.append(subpixels)

print(f"\nDots per distance: {[len(s) for s in subpixel_list]}")

# ── Visualise last calibration distance ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(image, cmap="gray", alpha=0.8)
for d in subpixels:
    ax.add_patch(plt.Circle((d["x"], d["y"]), 0.5,
                            edgecolor="red", facecolor="none", linewidth=1))
ax.set_title(f"Subpixel dot positions – {len(subpixels)} dots (last calibration distance)")
ax.axis("off")
plt.show()

## Calibration Step 3: 3-D Back-Projection

For each detected dot at pixel (u, v) and ToF depth d, compute the 3-D point
U_ij = d · K⁻¹ · [u, v, 1]ᵀ / ‖…‖  in camera coordinates.

In [ ]:
# Step 3: 3-D back-projection of calibration dots
BASELINE_GUESS = 3.8e-2   # initial transmitter x-offset estimate [m]

U, U_tx = dotCal.backproject_calibration_dots(
    subpixel_list, tof_paths,
    tof_mode="pcd", K=K,
    pcd_depth_mode="radial",
    baseline_guess=BASELINE_GUESS
)
print(f"U shape:    {U.shape}     (n_dots × n_dist × xyz)")
print(f"U_tx shape: {U_tx.shape}  (shifted by baseline_guess = {BASELINE_GUESS*100:.1f} cm)")

# ── Plot: SL images with detected dot IDs ──────────────────────────────
exr_files = sorted(sl_cal.glob("*.exr"),
                   key=lambda p: dotCal.parse_distance_from_name(p.name) or 999.0)
render_by_dist = {dotCal.parse_distance_from_name(p.name): str(p) for p in exr_files}

for j, dist in enumerate(cal_dists):
    exr = render_by_dist.get(dist)
    if exr is None:
        continue
    img = dotCal.read_image(exr)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(img, cmap="gray")
    for dot in subpixel_list[j]:
        ax.add_patch(plt.Circle((dot["x"], dot["y"]), 2,
                                edgecolor="red", facecolor="none", linewidth=1))
        ax.text(dot["x"] + 2, dot["y"] + 2, str(dot["id"]),
                fontsize=5, color="yellow")
    ax.set_title(f"{dist:.1f} m  |  {len(subpixel_list[j])} dots")
    ax.axis("off")
    plt.show()

## Calibration Step 4: Baseline Estimation

Fit a 3-D line through each dot's multi-distance measurements (in U_tx space),
then find the least-squares intersection of all lines.  The x-component is the
correction to the initial baseline guess, giving AB.

In [ ]:
# Step 4: baseline estimation
intersection, n_lines = dotCal.estimate_baseline(U_tx)
AB = BASELINE_GUESS + float(intersection[0])
B  = np.array([AB, 0.0, 0.0])

print(f"Fitted {n_lines} dot lines")
print(f"LS intersection (U_tx):  {intersection}")
print(f"Baseline AB:  {AB*100:.3f} cm   →  B = {B}")
print(f"y, z of intersection (expect ≈ 0):  "
      f"{float(intersection[1]):.4f}, {float(intersection[2]):.4f}")

# ── 3-D plot: points + lines + intersection ─────────────────────────────
all_pts = U_tx.reshape(-1, 3)
all_pts = all_pts[np.all(np.isfinite(all_pts), axis=1)]

fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection="3d")
ax.scatter(all_pts[:, 0], all_pts[:, 1], all_pts[:, 2],
           s=3, alpha=0.3, label="3-D calibration points (U_tx)")
ax.scatter(*intersection, s=200, marker="x", color="red",
           label=f"LS intersection  AB_corr={float(intersection[0])*100:.2f} cm")
ax.set_title("All 3-D calibration points in U_tx space + LS intersection")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
ax.legend()
plt.tight_layout()
plt.show()

## Calibration Step 5: Unit Vector Estimation

For each dot i, fit a unit direction vector V[i] by minimising the angle between
the fitted ray and each calibration point Q_ij = U_ij − B.

In [ ]:
# Step 5: unit vector estimation
V = dotCal.estimate_unit_vectors(U, AB)
V_valid = V[np.all(np.isfinite(V), axis=1)]
print(f"Unit vectors estimated: {len(V_valid)} / {V.shape[0]}")

# Angle-error diagnostics
angle_errs = []
for i in range(V.shape[0]):
    if not np.all(np.isfinite(V[i])):
        continue
    pts   = U[i, :, :]
    valid = np.all(np.isfinite(pts), axis=1)
    Q     = pts[valid] - B[None, :]
    Qn    = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)
    angle_errs.extend(np.degrees(np.arccos(np.clip(Qn @ V[i], -1, 1))).tolist())

# ── Quiver plot ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(9, 6))
ax  = fig.add_subplot(111, projection="3d")
ax.scatter([0], [0], [0], s=80, marker="o", label="Rx (camera)")
ax.scatter([B[0]], [B[1]], [B[2]], s=120, marker="x", color="red", label=f"Tx (AB={AB*100:.1f} cm)")
for i in range(0, V.shape[0], 10):
    if not np.all(np.isfinite(V[i])):
        continue
    ax.quiver(B[0], B[1], B[2], V[i, 0], V[i, 1], V[i, 2],
              length=0.4, normalize=True, alpha=0.5)
ax.set_title("Calibrated transmitter unit vectors V_i  (every 10th dot)")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
ax.legend()
plt.tight_layout()
plt.show()

# ── Angle-error histogram ─────────────────────────────────────────────
plt.figure(figsize=(7, 3))
plt.hist(np.array(angle_errs), bins=30, edgecolor="k", linewidth=0.4)
plt.title("Angle deviation between V_i and calibration points [°]")
plt.xlabel("Angle error [°]"); plt.ylabel("Count")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"Median angle error: {np.median(angle_errs):.3f}°")
print(f"Max    angle error: {np.max(angle_errs):.3f}°")

## Approach 1: Consistency Error  (Microsoft-Paper paper)

At test time, each detected dot is assigned to its closest calibration trail.
The trail is sampled at all calibration distances.  For each sample j along the
trail, both Z_ToF(j) and Z_triangulation(j) are computed.  The sample
j\* = argmin ε, with **ε = |1/Z_ToF − 1/Z_tri|**, is selected as the output depth.

In [ ]:
# ── Inputs ─────────────────────────────────────────────────────────────
SL_TEST      = str(test_dir / "Test_SL.exr")
TOF_TEST     = str(test_dir / "Test_ToF.pcd")
PCD_SCALE    = 0.001
EPS_THRESHOLD = None   # set a float to invalidate high-ε dots

# ── Load test scene ──────────────────────────────────────────────────────
trail_xy = dotCal.build_calibration_trails(subpixel_list)

tof_test   = dotCal.load_tof_pcd(TOF_TEST, unit_scale=PCD_SCALE, depth_mode="axial")
depth_map  = tof_test["depth_map"]
H, W       = depth_map.shape

blobs_test, sl_gray = dotCal.detect_blobs(
    SL_TEST, max_sigma=25, num_sigma=75, threshold=0.032, visualize=True
)
_, subpix_test = dotCal.detect_subpixel_locations(
    blobs_test, sl_gray, mode="geometricCenter"
)
test_uv = np.array([[d["x"], d["y"]] for d in subpix_test], dtype=float)
print(f"Detected dots in test SL: {len(subpix_test)}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(sl_gray, cmap="gray")
for d in subpix_test:
    ax.add_patch(plt.Circle((d["x"], d["y"]), 2,
                            edgecolor="red", facecolor="none", linewidth=1))
ax.set_title("Test SL: detected dots"); ax.axis("off"); plt.show()


# ── ToF depth sampler ────────────────────────────────────────────────────
def sample_tof_depth(u, v, search_radius=3):
    """Sample axial ToF depth from organised depth_map with local fill."""
    for r in range(search_radius + 1):
        for dv in range(-r, r + 1):
            for du in range(-r, r + 1):
                u2, v2 = int(round(u)) + du, int(round(v)) + dv
                if 0 <= u2 < W and 0 <= v2 < H:
                    z = depth_map[v2, u2]
                    if np.isfinite(z) and z > 1e-6:
                        return float(z), float(np.hypot(du, dv))
    return 1e-9, np.inf


# Trail KD-trees for fast dot-to-trail assignment
trail_trees = [cKDTree(trail_xy[:, i, :]) for i in range(100)]
n_dist = trail_xy.shape[0]


# ── Run Microsoft-Paper algorithm ──────────────────────────────────────────────────
results = []

for k, (uu, vv) in enumerate(test_uv):
    # a) assign dot to calibration ray (nearest trail)
    best_i, best_dist = min(
        enumerate(trail_trees),
        key=lambda t: t[1].query([uu, vv], k=1)[0]
    )
    best_dist = trail_trees[best_i].query([uu, vv], k=1)[0]

    # b) compute ε along all trail samples
    Z_tri_arr = np.empty(n_dist)
    Z_tof_arr = np.empty(n_dist)
    eps_arr   = np.empty(n_dist)

    for j in range(n_dist):
        uij = float(trail_xy[j, best_i, 0])
        vij = float(trail_xy[j, best_i, 1])
        Z_tri = dotCal.triangulate_depth(uij, vij, V[best_i], B, K_inv)
        Z_tof, _ = sample_tof_depth(uij, vij)
        Z_tri_arr[j] = max(Z_tri, 1e-9)
        Z_tof_arr[j] = max(Z_tof, 1e-9)
        eps_arr[j]   = dotCal.consistency_error(Z_tof_arr[j], Z_tri_arr[j])

    # c) & d) select min-ε sample
    j_best   = int(np.argmin(eps_arr))
    Z_raw, _ = sample_tof_depth(uu, vv)
    Z_out    = float(Z_tof_arr[j_best])
    if EPS_THRESHOLD is not None and float(eps_arr[j_best]) > EPS_THRESHOLD:
        Z_out = Z_raw

    results.append(dict(
        k=k, u=float(uu), v=float(vv),
        Z_raw=Z_raw, Z_out=Z_out,
        i_best=best_i, j_best=j_best,
        eps_best=float(eps_arr[j_best]),
        eps_curve=eps_arr.copy(),
        Z_tri_curve=Z_tri_arr.copy(),
        Z_tof_curve=Z_tof_arr.copy(),
    ))

print(f"Processed {len(results)} dots")


# ── ε-curve plots (best and worst dot) ──────────────────────────────────
eps_all = np.array([r["eps_best"] for r in results], float)
for label, k_ex in [("best ε", int(np.argmin(eps_all))),
                     ("worst ε", int(np.argmax(eps_all)))]:
    ex  = results[k_ex]
    x   = np.arange(n_dist)
    eps_plot = ex["eps_curve"] * 1e-3   # 1/m → 1/mm (paper scale)

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
    axes[0].plot(x, ex["Z_tof_curve"], "*", markersize=5)
    axes[0].set_title("ToF depth along trail")
    axes[0].set_xlabel("Trail sample j"); axes[0].set_ylabel("Z [m]")

    axes[1].plot(x, 1 / ex["Z_tof_curve"], "*", markersize=5, label="1/Z_ToF")
    axes[1].plot(x, 1 / ex["Z_tri_curve"], "*", markersize=5, label="1/Z_tri")
    axes[1].set_title("1/Z comparison (disparity space)")
    axes[1].set_xlabel("Trail sample j"); axes[1].set_ylabel("1/Z [1/m]")
    axes[1].legend(fontsize=8)

    axes[2].semilogy(x, eps_plot, "*", markersize=5, color="C2")
    axes[2].axvline(ex["j_best"], color="red", linewidth=1.2, linestyle="--",
                    label=f"j*={ex['j_best']}")
    axes[2].set_title(f"Consistency error ε  ({label})")
    axes[2].set_xlabel("Trail sample j")
    axes[2].set_ylabel(r"$\varepsilon$ [mm$^{-1}$]")
    axes[2].legend(fontsize=8)

    plt.suptitle(
        f"Microsoft-Paper Figure 8 – dot k={ex['k']}  (i*={ex['i_best']}, "
        f"ε*={ex['eps_best']:.3e})", y=1.02
    )
    plt.tight_layout(); plt.show()


# ── Depth per dot: raw vs approach 1 ────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot([r["k"] for r in results], [r["Z_raw"] for r in results],
         ".", markersize=5, label="Z_ToF (raw @ detected dot)")
plt.plot([r["k"] for r in results], [r["Z_out"] for r in results],
         ".", markersize=5, label="Z_out (ToF @ min-ε trail sample)")
plt.title("Approach 1 – Consistency error: raw ToF vs selected depth")
plt.xlabel("Dot index k"); plt.ylabel("Depth [m]")
plt.legend(); plt.grid(alpha=0.3); plt.show()


# ── Depth overlay on SL image ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(sl_gray, cmap="gray")
xs = np.array([r["u"] for r in results])
ys = np.array([r["v"] for r in results])
zs = np.array([r["Z_out"] for r in results])
sc = ax.scatter(xs, ys, s=40, c=zs, cmap="plasma", alpha=0.9)
plt.colorbar(sc, ax=ax, label="Z_out [m]")
for r in results:
    ax.text(r["u"] + 2, r["v"] + 2,  f'{r["Z_out"]:.2f}', fontsize=5, color="yellow")
    ax.text(r["u"] + 2, r["v"] + 12, f'i={r["i_best"]}',  fontsize=5, color="cyan")
ax.set_title("Approach 1 – Consistency error: depth overlay")
ax.axis("off"); plt.show()


# ── ε histogram ─────────────────────────────────────────────────────────
plt.figure(figsize=(6, 3))
plt.hist(eps_all, bins=30, edgecolor="k", linewidth=0.4)
if EPS_THRESHOLD:
    plt.axvline(EPS_THRESHOLD, color="red", label=f"threshold={EPS_THRESHOLD:.3e}")
    plt.legend()
plt.title(r"Distribution of best consistency errors $\varepsilon = |1/Z_{ToF} - 1/Z_{tri}|$")
plt.xlabel(r"$\varepsilon$ [1/m]"); plt.ylabel("Count")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Approach 2: Maximum Likelihood Fusion  (Agresti & Zanuttigh, ECCV 2018)

ToF and SL (triangulation) depth estimates are combined via maximum likelihood
using spatially-weighted mixture-of-Gaussians likelihoods derived from per-pixel
noise variance models.  The fused depth maximises the joint log-likelihood.

In [ ]:
# Uses sl_gray, depth_map, trail_xy, test_uv, V, B, K_inv from Approach 1 cell.

# ── Settings ────────────────────────────────────────────────────────────
MAX_TOF_PIX_DIST = 6.0
TOF_FILL_WIN     = 2
TRAIL_SOFT_SCALE = 6.0    # px – inflates σ_SL when dot is far from trail
wh               = 3      # spatial neighbourhood half-width
Z_CAND_N         = 200    # number of depth candidates for ML search

# ── Load test ToF with 3-D point map ────────────────────────────────────
tof_test_ml = dotCal.load_tof_pcd(TOF_TEST, unit_scale=PCD_SCALE, depth_mode="axial")
depth_map_ml = tof_test_ml["depth_map"]
points_map   = tof_test_ml["points_3d"].reshape(H, W, 3)


# ── Samplers ─────────────────────────────────────────────────────────────
def sample_tof_ml(u, v):
    """Return (range, 3D-point, pixel-distance) from organised depth+point maps."""
    uu, vv = int(round(u)), int(round(v))
    for r in range(TOF_FILL_WIN + 1):
        for dv in range(-r, r + 1):
            for du in range(-r, r + 1):
                u2, v2 = uu + du, vv + dv
                if 0 <= u2 < W and 0 <= v2 < H:
                    z = depth_map_ml[v2, u2]
                    P = points_map[v2, u2]
                    if np.isfinite(z) and z > 1e-6 and np.all(np.isfinite(P)):
                        return float(z), P.astype(float), float(np.hypot(du, dv))
    return np.nan, None, np.inf


def cam_ray_from_hint(u, v, P_hint=None):
    """Camera ray – uses 3-D PCD point if available, falls back to pinhole."""
    if P_hint is not None and np.all(np.isfinite(P_hint)):
        n = np.linalg.norm(P_hint)
        if n > 1e-9:
            return P_hint / n
    return DotCalibration.cam_ray(u, v, K_inv)


def triangulate_range(u, v, v_i, P_hint=None):
    """Triangulate scalar range along the camera ray."""
    r   = cam_ray_from_hint(u, v, P_hint)
    vi  = v_i / (np.linalg.norm(v_i) + 1e-12)
    A   = np.column_stack([r, -vi])
    st, *_ = np.linalg.lstsq(A, B, rcond=None)
    return float(st[0])


# ── Assign trail and compute per-dot depths ──────────────────────────────
test_ztof = np.full(len(test_uv), np.nan)
test_P3d  = [None] * len(test_uv)
test_nnpx = np.full(len(test_uv), np.nan)

for k, (uu, vv) in enumerate(test_uv):
    z, P, dpx       = sample_tof_ml(uu, vv)
    test_ztof[k]    = z
    test_P3d[k]     = P
    test_nnpx[k]    = dpx

z_sl      = np.full(len(test_uv), np.nan)
i_best_ml = np.full(len(test_uv), -1, int)
trail_d   = np.full(len(test_uv), np.nan)

for k, (uu, vv) in enumerate(test_uv):
    i, dpx = min(
        enumerate(trail_trees),
        key=lambda t: t[1].query([uu, vv], k=1)[0]
    )
    dpx = trail_trees[i].query([uu, vv], k=1)[0]
    i_best_ml[k] = i
    trail_d[k]   = dpx
    z = triangulate_range(uu, vv, V[i], P_hint=test_P3d[k])
    if np.isfinite(z) and z > 1e-6:
        z_sl[k] = z

print(f"ToF valid: {int(np.sum(np.isfinite(test_ztof)))} / {len(test_uv)}",
      f"  range: {np.nanmin(test_ztof):.2f}–{np.nanmax(test_ztof):.2f} m")
print(f"SL  valid: {int(np.sum(np.isfinite(z_sl)))} / {len(test_uv)}",
      f"  range: {np.nanmin(z_sl):.2f}–{np.nanmax(z_sl):.2f} m")


# ── Uncertainty models ───────────────────────────────────────────────────
def robust_sigma(vals):
    vals = np.asarray(vals, float)
    vals = vals[np.isfinite(vals)]
    if vals.size < 3:
        return np.nan
    return 1.4826 * np.median(np.abs(vals - np.median(vals)))

sigma_tof = np.full(len(test_uv), np.nan)
sigma_sl  = np.full(len(test_uv), np.nan)

for k, (uu, vv) in enumerate(test_uv):
    if np.isfinite(test_ztof[k]) and test_ztof[k] > 1e-6:
        ui, vi = int(round(uu)), int(round(vv))
        r = 3
        patch = depth_map_ml[max(0, vi-r):vi+r+1, max(0, ui-r):ui+r+1].ravel()
        s_loc = robust_sigma(patch)
        if not np.isfinite(s_loc):
            s_loc = 0.02
        sigma_tof[k] = max(0.01, s_loc + 0.02 * (test_nnpx[k] / max(MAX_TOF_PIX_DIST, 1e-6))**2)

    if np.isfinite(z_sl[k]) and z_sl[k] > 1e-6:
        base = 0.01 + 0.06 * (z_sl[k] / 4.0)**2
        if np.isfinite(trail_d[k]):
            base *= 1.0 + (trail_d[k] / max(TRAIL_SOFT_SCALE, 1e-6))**2
        sigma_sl[k] = max(0.01, base)


# ── ML Fusion (Agresti eq. 16-17) ───────────────────────────────────────
dot_tree = cKDTree(test_uv)
if len(test_uv) >= 5:
    dd, _ = dot_tree.query(test_uv, k=2)
    sigma_s_px = 0.5 * np.median(dd[:, 1])
else:
    sigma_s_px = 10.0


def log_mog_likelihood(Zcand, depths, sigmas, weights):
    """Log mixture-of-Gaussians likelihood over depth candidates."""
    m = np.isfinite(depths) & np.isfinite(sigmas) & (sigmas > 1e-9) & (weights > 0)
    if not np.any(m):
        return np.full_like(Zcand, -np.inf)
    d, s, w = depths[m], sigmas[m], weights[m]
    log_pref = np.log(np.maximum(w, 1e-12)) - np.log(s)
    out = np.empty(len(Zcand))
    for t, Z0 in enumerate(Zcand):
        lc  = log_pref - 0.5 * ((d - Z0) / s)**2
        mx  = np.max(lc)
        out[t] = mx + np.log(np.sum(np.exp(lc - mx)) + 1e-12)
    return out


knn_patch = (2 * wh + 1)**2
z_fus     = np.full(len(test_uv), np.nan)

for k in range(len(test_uv)):
    zt, zs = test_ztof[k], z_sl[k]
    st, ss = sigma_tof[k], sigma_sl[k]

    if np.isfinite(zs) and not np.isfinite(zt):
        z_fus[k] = float(zs); continue
    if np.isfinite(zt) and not np.isfinite(zs):
        z_fus[k] = float(zt); continue
    if not (np.isfinite(zt) and np.isfinite(zs) and np.isfinite(st) and np.isfinite(ss)):
        continue

    dnn, nn_idx = dot_tree.query(test_uv[k], k=min(knn_patch, len(test_uv)))
    w_sp = np.exp(-0.5 * (dnn / max(sigma_s_px, 1e-6))**2)

    zmin = max(1e-4, min(zt - 3*st, zs - 3*ss))
    zmax = max(zt + 3*st, zs + 3*ss)
    if not (np.isfinite(zmin) and np.isfinite(zmax)) or zmax <= zmin:
        continue

    Zcand   = np.linspace(zmin, zmax, Z_CAND_N)
    ll_tof  = log_mog_likelihood(Zcand, test_ztof[nn_idx], sigma_tof[nn_idx], w_sp)
    ll_sl   = log_mog_likelihood(Zcand, z_sl[nn_idx],  sigma_sl[nn_idx],  w_sp)
    z_fus[k] = float(Zcand[int(np.argmax(ll_tof + ll_sl))])

valid = np.isfinite(z_fus)
print(f"Fused valid: {int(np.sum(valid))} / {len(test_uv)}",
      f"  range: {np.nanmin(z_fus):.2f}–{np.nanmax(z_fus):.2f} m")


# ── Depth overlay ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(sl_gray, cmap="gray")
xs, ys = test_uv[:, 0], test_uv[:, 1]
sc = ax.scatter(xs[valid], ys[valid], s=45, c=z_fus[valid], cmap="plasma", alpha=0.9)
plt.colorbar(sc, ax=ax, label="Fused depth [m]")
for k in np.where(valid)[0]:
    ax.text(xs[k] + 2, ys[k] + 2,  f"{z_fus[k]:.2f}", fontsize=5, color="yellow")
    ax.text(xs[k] + 2, ys[k] + 12, f"i={i_best_ml[k]}", fontsize=5, color="cyan")
ax.set_title("Approach 2 – ML fusion: depth overlay"); ax.axis("off"); plt.show()


# ── Per-dot depth comparison ─────────────────────────────────────────────
m = (i_best_ml >= 0) & (np.isfinite(test_ztof) | np.isfinite(z_sl) | np.isfinite(z_fus))
order = np.argsort(i_best_ml[m])
x = i_best_ml[m][order]

plt.figure(figsize=(11, 4))
mt = np.isfinite(test_ztof[m][order])
ms = np.isfinite(z_sl[m][order])
mf = np.isfinite(z_fus[m][order])
plt.plot(x[mt], test_ztof[m][order][mt], "o", markersize=3, label="ToF")
plt.plot(x[ms], z_sl[m][order][ms],       "x", markersize=3, label="SL (triangulation)")
plt.plot(x[mf], z_fus[m][order][mf],      ".", markersize=6, label="ML fused")
plt.xlabel("Matched ray index i"); plt.ylabel("Depth [m]")
plt.ylim(0, 5)
plt.title("Approach 2 – ML fusion: ToF vs SL vs Fused (sorted by ray index)")
plt.grid(alpha=0.3); plt.legend(); plt.show()


# ── Likelihood visualisation for one example dot ─────────────────────────
valid_k = [k for k in range(len(test_uv))
           if np.isfinite(test_ztof[k]) and np.isfinite(z_sl[k]) and np.isfinite(z_fus[k])]

if valid_k:
    k_ex = valid_k[min(42, len(valid_k) - 1)]
    zt, zs = test_ztof[k_ex], z_sl[k_ex]
    st, ss = sigma_tof[k_ex], sigma_sl[k_ex]

    dnn, nn_idx = dot_tree.query(test_uv[k_ex], k=min(knn_patch, len(test_uv)))
    w_sp = np.exp(-0.5 * (dnn / max(sigma_s_px, 1e-6))**2)

    zmin_p = max(1e-4, min(zt - 4*st, zs - 4*ss))
    zmax_p = max(zt + 4*st, zs + 4*ss)
    Zp     = np.linspace(zmin_p, zmax_p, 500)

    ll_t = log_mog_likelihood(Zp, test_ztof[nn_idx], sigma_tof[nn_idx], w_sp)
    ll_s = log_mog_likelihood(Zp, z_sl[nn_idx],  sigma_sl[nn_idx],  w_sp)
    ll_j = ll_t + ll_s

    prob_t = np.exp(ll_t - np.max(ll_t))
    prob_s = np.exp(ll_s - np.max(ll_s))
    prob_j = np.exp(ll_j - np.max(ll_j))
    z_max  = float(Zp[int(np.argmax(prob_j))])

    plt.figure(figsize=(10, 5))
    plt.plot(Zp, prob_t, label="Likelihood ToF",      color="C0", linewidth=2, alpha=0.7)
    plt.plot(Zp, prob_s, label="Likelihood SL",       color="C1", linewidth=2, alpha=0.7)
    plt.plot(Zp, prob_j, label="Joint likelihood (fused)", color="C2", linewidth=3)
    plt.axvline(zt,    color="C0", linestyle="--", alpha=0.5, label=f"Z_ToF = {zt:.3f} m")
    plt.axvline(zs,    color="C1", linestyle="--", alpha=0.5, label=f"Z_SL  = {zs:.3f} m")
    plt.axvline(z_max, color="red", linewidth=2, label=f"Z_fused = {z_max:.3f} m")
    plt.title(f"Approach 2 – ML likelihood curves for dot k={k_ex} (normalised)")
    plt.xlabel("Depth candidate Z [m]"); plt.ylabel("Relative likelihood")
    plt.legend(loc="upper right"); plt.grid(alpha=0.3)
    plt.xlim([zmin_p, zmax_p]); plt.tight_layout(); plt.show()

## Approach 3: Active Brightness Trail  (Microsoft-Paper paper)

Instead of 2-D blob detection on the SL image, the iToF **active brightness**
(signal amplitude = grayValue channel of the ToF point cloud) is resampled along
each pre-computed 1-D calibration trail.  Peaks in the brightness profile are
candidate dot positions.  For each peak the consistency error
ε = |1/Z_ToF − 1/Z_tri| is evaluated and the peak with minimum ε selects the
final depth measurement.  This approach is robust to scene texture and low SNR
because detection is constrained to a 1-D path rather than a 2-D image.

In [ ]:
# Uses trail_xy, V, B, K_inv, depth_map (from Approach 1)

# ── Settings ────────────────────────────────────────────────────────────
N_TRAIL_DENSE  = 300     # dense interpolation steps along each trail
PEAK_PROM      = 0.05    # scipy.signal.find_peaks prominence (relative)
PEAK_DIST      = 5       # minimum peak separation in trail samples


# ── Load active brightness image from test ToF PCD ───────────────────────
tof_ab = dotCal.load_tof_pcd(TOF_TEST, unit_scale=PCD_SCALE, depth_mode="axial")
brightness_map = tof_ab["intensity_map"]   # (H, W) grayValue
depth_map_ab   = tof_ab["depth_map"]        # (H, W) axial Z

# Normalise brightness for easier peak detection
bmax = np.nanmax(brightness_map)
if bmax > 0:
    brightness_norm = brightness_map / bmax
else:
    brightness_norm = brightness_map.copy()


# ── Bilinear sampler ────────────────────────────────────────────────────
def bilinear_sample(img, u, v):
    """Bilinear interpolation of img at sub-pixel position (u, v)."""
    u0, v0 = int(np.floor(u)), int(np.floor(v))
    u1, v1 = u0 + 1, v0 + 1
    if u0 < 0 or v0 < 0 or u1 >= img.shape[1] or v1 >= img.shape[0]:
        return np.nan
    du, dv = u - u0, v - v0
    return (img[v0, u0] * (1 - du) * (1 - dv) +
            img[v0, u1] * du * (1 - dv) +
            img[v1, u0] * (1 - du) * dv +
            img[v1, u1] * du * dv)


# ── Dense trail interpolation ────────────────────────────────────────────
# trail_xy: (n_dist, n_dots, 2)  – pixel coords at discrete calibration distances
n_dist_cal = trail_xy.shape[0]
j_sparse   = np.arange(n_dist_cal, dtype=float)
j_dense    = np.linspace(0, n_dist_cal - 1, N_TRAIL_DENSE)

# For each dot, build dense (u, v) trail by cubic spline (falls back to linear)
trail_dense_uv = np.empty((N_TRAIL_DENSE, 100, 2))  # (samples, dots, uv)
for i in range(100):
    trail_dense_uv[:, i, 0] = np.interp(j_dense, j_sparse, trail_xy[:, i, 0])
    trail_dense_uv[:, i, 1] = np.interp(j_dense, j_sparse, trail_xy[:, i, 1])


# ── Run Active Brightness Trail algorithm ────────────────────────────────
ab_results = []

for i in range(100):
    if not np.all(np.isfinite(V[i])):
        continue

    u_trail = trail_dense_uv[:, i, 0]
    v_trail = trail_dense_uv[:, i, 1]

    # Sample brightness along trail
    brightness_profile = np.array([
        bilinear_sample(brightness_norm, u, v)
        for u, v in zip(u_trail, v_trail)
    ])
    brightness_profile = np.nan_to_num(brightness_profile, nan=0.0)

    # Detect peaks in brightness profile
    peaks, peak_props = scipy.signal.find_peaks(
        brightness_profile,
        prominence=PEAK_PROM * np.max(brightness_profile) if np.max(brightness_profile) > 0 else 0.01,
        distance=PEAK_DIST
    )

    if len(peaks) == 0:
        # Fall back to global maximum if no peak found
        peaks = np.array([int(np.argmax(brightness_profile))])

    # For each peak: compute Z_ToF and Z_tri, then ε
    peak_eps    = []
    peak_Z_tof  = []
    peak_Z_tri  = []

    for pk in peaks:
        u_pk, v_pk = float(u_trail[pk]), float(v_trail[pk])
        Z_tri = dotCal.triangulate_depth(u_pk, v_pk, V[i], B, K_inv)
        Z_tof, _ = sample_tof_depth(u_pk, v_pk)
        eps   = dotCal.consistency_error(max(Z_tof, 1e-9), max(Z_tri, 1e-9))
        peak_eps.append(eps)
        peak_Z_tof.append(max(Z_tof, 1e-9))
        peak_Z_tri.append(max(Z_tri, 1e-9))

    j_best_pk = int(np.argmin(peak_eps))
    pk_best   = peaks[j_best_pk]
    u_out     = float(u_trail[pk_best])
    v_out     = float(v_trail[pk_best])

    ab_results.append(dict(
        dot_id           = i,
        u                = u_out,
        v                = v_out,
        Z_out            = peak_Z_tof[j_best_pk],
        Z_tri            = peak_Z_tri[j_best_pk],
        eps_best         = peak_eps[j_best_pk],
        n_peaks          = len(peaks),
        brightness_profile = brightness_profile.copy(),
        peaks            = peaks.copy(),
        peak_eps         = np.array(peak_eps),
        peak_Z_tof       = np.array(peak_Z_tof),
        peak_Z_tri       = np.array(peak_Z_tri),
        u_trail          = u_trail.copy(),
        v_trail          = v_trail.copy(),
    ))

print(f"Active brightness results: {len(ab_results)} dots")
print(f"Dots with >1 peak (MPI indicator): "
      f"{sum(r['n_peaks'] > 1 for r in ab_results)}")


# ── Plot 1: Active brightness image with trail overlays ──────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(brightness_norm, cmap="hot", vmin=0, vmax=1)
for r in ab_results[::5]:   # every 5th trail
    ax.plot(r["u_trail"], r["v_trail"], "-", color="cyan", linewidth=0.4, alpha=0.5)
for r in ab_results:
    ax.scatter(r["u"], r["v"], s=15, color="red", zorder=5)
ax.set_title("Active Brightness Image with calibration trails and selected peaks")
ax.set_xlabel("u [px]"); ax.set_ylabel("v [px]")
plt.tight_layout(); plt.show()


# ── Plot 2: brightness profiles for best / worst ε dots ──────────────────
eps_ab = np.array([r["eps_best"] for r in ab_results], float)
for label, idx in [("best ε", int(np.argmin(eps_ab))),
                    ("worst ε", int(np.argmax(eps_ab)))]:
    r  = ab_results[idx]
    xp = np.arange(N_TRAIL_DENSE)

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))

    # Brightness profile + detected peaks
    axes[0].plot(xp, r["brightness_profile"], linewidth=1)
    axes[0].plot(r["peaks"], r["brightness_profile"][r["peaks"]],
                 "v", color="red", markersize=8, label="peaks")
    axes[0].axvline(r["peaks"][int(np.argmin(r["peak_eps"]))],
                   color="green", linewidth=1.5, linestyle="--", label="selected")
    axes[0].set_title(f"Brightness profile along trail  ({label})")
    axes[0].set_xlabel("Dense trail index"); axes[0].set_ylabel("Brightness (norm.)")
    axes[0].legend(fontsize=8)

    # Z comparison at peak positions
    pk_x = np.arange(len(r["peaks"]))
    axes[1].plot(pk_x, r["peak_Z_tof"], "o-", markersize=6, label="Z_ToF")
    axes[1].plot(pk_x, r["peak_Z_tri"], "s-", markersize=6, label="Z_tri")
    axes[1].axvline(int(np.argmin(r["peak_eps"])), color="green", linewidth=1.5,
                   linestyle="--", label="selected")
    axes[1].set_title("Z_ToF and Z_tri at detected peaks")
    axes[1].set_xlabel("Peak index"); axes[1].set_ylabel("Depth [m]")
    axes[1].legend(fontsize=8)

    # ε at each peak
    axes[2].semilogy(pk_x, r["peak_eps"] * 1e-3, "D-", markersize=7, color="C2")
    axes[2].axvline(int(np.argmin(r["peak_eps"])), color="green", linewidth=1.5,
                   linestyle="--", label=f"min-ε  ({r['eps_best']:.3e})")
    axes[2].set_title(r"Consistency error $\varepsilon$ per peak")
    axes[2].set_xlabel("Peak index")
    axes[2].set_ylabel(r"$\varepsilon$ [mm$^{-1}$]")
    axes[2].legend(fontsize=8)

    plt.suptitle(
        f"Active Brightness Trail – dot i={r['dot_id']}  ({label})\n"
        f"n_peaks={r['n_peaks']}, Z_out={r['Z_out']:.3f} m, ε*={r['eps_best']:.3e}",
        y=1.04
    )
    plt.tight_layout(); plt.show()


# ── Plot 3: depth overlay on SL image ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(sl_gray, cmap="gray")
xs_ab = np.array([r["u"]     for r in ab_results])
ys_ab = np.array([r["v"]     for r in ab_results])
zs_ab = np.array([r["Z_out"] for r in ab_results])
sc = ax.scatter(xs_ab, ys_ab, s=40, c=zs_ab, cmap="plasma", alpha=0.9)
plt.colorbar(sc, ax=ax, label="Z_out [m]")
for r in ab_results:
    ax.text(r["u"] + 2, r["v"] + 2,  f'{r["Z_out"]:.2f}', fontsize=5, color="yellow")
    ax.text(r["u"] + 2, r["v"] + 12, f'i={r["dot_id"]}',  fontsize=5, color="cyan")
ax.set_title("Approach 3 – Active Brightness Trail: depth overlay")
ax.axis("off"); plt.show()


# ── Plot 4: depth comparison across all three approaches ─────────────────
# Build lookup by dot_id for approach 3
ab_by_id = {r["dot_id"]: r["Z_out"] for r in ab_results}

# Collect from Approach 1 results (keyed by i_best)
approach1_by_i = {}
for r in results:
    i = r["i_best"]
    approach1_by_i.setdefault(i, []).append(r["Z_out"])

dot_ids_common = sorted(ab_by_id.keys())
z_ab  = np.array([ab_by_id[i] for i in dot_ids_common])
z_ap1 = np.array([np.mean(approach1_by_i[i]) if i in approach1_by_i else np.nan
                  for i in dot_ids_common])

plt.figure(figsize=(11, 4))
plt.plot(dot_ids_common, z_ab, "o", markersize=4, label="Approach 3 – Active Brightness")
m1 = np.isfinite(z_ap1)
plt.plot(np.array(dot_ids_common)[m1], z_ap1[m1], "x", markersize=5,
         label="Approach 1 – Consistency error")
plt.xlabel("Dot index i"); plt.ylabel("Depth [m]")
plt.title("Depth comparison: Approach 1 (Consistency Error) vs Approach 3 (Active Brightness Trail)")
plt.grid(alpha=0.3); plt.legend(); plt.show()


# ── Plot 5: ε histogram ──────────────────────────────────────────────────
plt.figure(figsize=(6, 3))
plt.hist(eps_ab, bins=30, edgecolor="k", linewidth=0.4)
plt.title("Approach 3 – Active Brightness Trail: best ε per dot")
plt.xlabel(r"$\varepsilon$ [1/m]"); plt.ylabel("Count")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"Median ε: {np.median(eps_ab):.4e} 1/m")
print(f"Depth range: {np.min(zs_ab):.2f}–{np.max(zs_ab):.2f} m")